In [ ]:
import os, sys
sys.path.append('..')
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()

## 1. Estructura de los Mensajes

La API usa una lista de mensajes con roles alternados `user` / `assistant`:

```python
messages = [
    {"role": "user",      "content": "Hola"},
    {"role": "assistant", "content": "Hola, ¿en qué puedo ayudarte?"},
    {"role": "user",      "content": "¿Cuál es la capital de Francia?"},
]
```

In [ ]:
# Conversación manual: primer turno
conversation = []

def chat(user_message: str, model: str = "claude-sonnet-4-5", max_tokens: int = 512) -> str:
    """Envía un mensaje y actualiza el historial de la conversación."""
    conversation.append({"role": "user", "content": user_message})
    
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=conversation
    )
    
    assistant_text = response.content[0].text
    conversation.append({"role": "assistant", "content": assistant_text})
    return assistant_text

# Primer mensaje
resp1 = chat("Mi nombre es Ana y estoy aprendiendo Python.")
print("Claude:", resp1)

In [ ]:
# Segundo turno — Claude debe recordar el nombre
resp2 = chat("¿Recuerdas cómo me llamo?")
print("Claude:", resp2)

In [ ]:
# Tercer turno
resp3 = chat("¿Qué lenguaje estaba aprendiendo?")
print("Claude:", resp3)

In [ ]:
# Ver el historial completo
print("=== Historial de la conversacion ===")
for i, msg in enumerate(conversation):
    role = "Usuario" if msg["role"] == "user" else "Asistente"
    print(f"\n[{i+1}] {role}:")
    print(f"   {msg['content'][:150]}")


## 2. Chat Interactivo en Terminal

El siguiente bloque implementa un chat interactivo en la consola. Escribe `salir` para terminar.

In [ ]:
def interactive_chat():
    """Chat interactivo en la consola del notebook."""
    history = []
    print("Chat con Claude -- escribe 'salir' para terminar\n")
    
    while True:
        user_input = input("Tu: ").strip()
        if not user_input or user_input.lower() in ("salir", "exit", "quit"):
            print("\nHasta luego!")
            break
        
        history.append({"role": "user", "content": user_input})
        
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1024,
            messages=history
        )
        
        assistant_reply = response.content[0].text
        history.append({"role": "assistant", "content": assistant_reply})
        print(f"\nClaude: {assistant_reply}\n")

# Descomenta para usar el chat interactivo:
# interactive_chat()


## 3. Ejercicio: Chat con Memoria

**Tarea**: Modifica la función `chat` para que:
1. Acepte un parámetro `max_history` que limite cuántos turnos anteriores se envían
2. Imprima cuántos tokens se usaron en cada respuesta

In [ ]:
# Tu solución aquí
def chat_with_memory(user_message: str, history: list, max_history: int = 10,
                     model: str = "claude-sonnet-4-5") -> tuple[str, list]:
    """
    Envía un mensaje usando solo los últimos `max_history` turnos del historial.
    Retorna (respuesta_texto, historial_actualizado)
    """
    # Añadir el nuevo mensaje
    history.append({"role": "user", "content": user_message})
    
    # Limitar el historial (mantener pares user/assistant)
    truncated = history[-(max_history * 2):]
    
    response = client.messages.create(
        model=model,
        max_tokens=512,
        messages=truncated
    )
    
    assistant_text = response.content[0].text
    history.append({"role": "assistant", "content": assistant_text})
    
    print(f"   [tokens: {response.usage.input_tokens} in / {response.usage.output_tokens} out]")
    return assistant_text, history

# Prueba
hist = []
resp, hist = chat_with_memory("Me llamo Carlos y trabajo como médico.", hist)
print("Claude:", resp)
resp, hist = chat_with_memory("¿Cuál es mi profesión?", hist)
print("Claude:", resp)